# EdNet 전체 데이터 테이블 만들기

01에서 파악한 구조를 바탕으로, **전체 유저 CSV를 DuckDB 테이블로** 적재합니다.

## 1. KT1 테이블 생성
- `*.csv`: 표본 5명이 아닌 전체 유저 파일을 읽음
- `parse_filename(filename, true)`: 파일 경로에서 `u1` 같은 유저 ID만 뽑아 `user_id` 컬럼으로 저장

In [1]:
import duckdb

con = duckdb.connect("../data/ednet.duckdb")

con.sql("""
CREATE OR REPLACE TABLE kt1 AS
SELECT
    parse_filename(filename, true) AS user_id,
    * EXCLUDE (filename)
FROM read_csv('../data/KT1/*.csv', filename = true)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. 적재 결과 확인
`filename` 대신 `user_id`가 맨 앞 컬럼으로 들어갔는지 확인합니다.

In [2]:
con.sql("""
SELECT *
FROM kt1
LIMIT 10
""").df()

,user_id,timestamp,solving_id,question_id,user_answer,elapsed_time
0,u1,1565096190868,1,q5012,b,38000
1,u1,1565096221062,2,q4706,c,24000
2,u1,1565096293432,3,q4366,b,68000
3,u1,1565096339668,4,q4829,a,42000
4,u1,1565096401774,5,q6528,b,59000
5,u1,1565096463370,6,q4793,a,58000
6,u1,1565096501746,7,q6488,a,35000
7,u1,1565097101361,8,q356,b,23000
8,u1,1565097171393,9,q1382,c,22000
9,u1,1565097240758,10,q830,b,25000


## 3. `user_id` 검증
빈 값이 없는지, 모두 `u숫자` 형식인지 확인합니다 → 이상 없음, 유저 784,309명

In [7]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE user_id IS NULL) AS null_count,
    COUNT(*) FILTER (WHERE NOT regexp_matches(user_id, '^u[0-9]+$')) AS bad_format_count,
    COUNT(DISTINCT user_id) AS user_count
FROM kt1
""").df()

,null_count,bad_format_count,user_count
0,0,0,784309


## 4. 유저별 푼 문제 수 분포
중앙값은 10문제지만 평균은 약 102문제, 최대 11,466문제로 **소수의 헤비 유저에 크게 치우친 분포**입니다.

In [13]:
summary = con.sql("""
WITH per_user AS (
    SELECT
        user_id,
        COUNT(DISTINCT question_id) AS solved_count
    FROM kt1
    GROUP BY user_id
)
SELECT
    MIN(solved_count) AS min_solved,
    quantile_cont(solved_count, 0.25) AS q1,
    median(solved_count) AS median_solved,
    quantile_cont(solved_count, 0.75) AS q3,
    MAX(solved_count) AS max_solved,
    AVG(solved_count) AS avg_solved
FROM per_user
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_solved,q1,median_solved,q3,max_solved,avg_solved
0,1,7.0,10.0,37.0,11466,101.911324
